# Signal Reconstruction
### **Purpose:** Compare EMG signal reconstruction quality across three methods.
# 
### **The Encode-Decode Pipeline:**
1. **Encode (Compress):** Reduce the 640-dimensional EMG window to a low-dimensional representation
2. **Decode (Reconstruct):** Rebuild the original 640-dimensional signal from the compressed form
3. **Measure:** Compare original vs reconstructed using RMSE, R², SNR
# 
### **Methods:**
1. MiniBatch Dictionary Learning (sparse codes → weighted atom sum)
2. K-SVD Dictionary Learning (sparse codes → weighted atom sum)
3. Classical Features (pseudo-inverse reconstruction, for comparison)

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('../')

import os
import numpy as np
import matplotlib.pyplot as plt
from src.config import DATABASE_CONFIGS
from src.reconstruction import run_all_reconstruction_experiments
from src.visualization import (
    plot_reconstruction_comparison,
    plot_reconstruction_metrics_comparison,
    plot_sparse_code_heatmap,
    plot_dictionary_atoms
)

## CONFIGURATION

In [ ]:
DATABASE = 'DB5'
SUBJECT = 1
LOAD_DIR = f'../data/preprocessed/{DATABASE}/'
prefix = f'S{SUBJECT:02d}'

# Dictionary parameters
N_ATOMS = 100       # Compression dimension (number of dictionary atoms)
N_NONZERO = 5       # Sparsity level (how many atoms used per window)
KSVD_ITER = 10      # K-SVD iterations

config = DATABASE_CONFIGS[DATABASE]

print(f"Signal Reconstruction: {DATABASE} Subject {SUBJECT}")
print(f"Original dimension: {int(config['window_ms']/1000*config['fs'])} samples × {config['n_channels']} channels = {int(config['window_ms']/1000*config['fs'])*config['n_channels']}")
print(f"Compression: {N_ATOMS} atoms, sparsity k={N_NONZERO}")

## 1. Load preprocessed data

In [ ]:
print("\nLoading data...")
X_train = np.load(os.path.join(LOAD_DIR, f'{prefix}_X_train.npy'))
X_test = np.load(os.path.join(LOAD_DIR, f'{prefix}_X_test.npy'))

# Use a subset of test data for faster experimentation
N_TEST = min(500, X_test.shape[0])
X_test_subset = X_test[:N_TEST]

print(f"Train: {X_train.shape}")
print(f"Test (subset): {X_test_subset.shape}")

## 2. Run reconstruction experiments

In [ ]:
print("\n" + "="*60)
print("RUNNING RECONSTRUCTION EXPERIMENTS")
print("="*60)

results = run_all_reconstruction_experiments(
    X_train, X_test_subset,
    n_atoms=N_ATOMS,
    n_nonzero=N_NONZERO,
    ksvd_max_iter=KSVD_ITER
)

## 3. VISUAL COMPARISON: Original vs Reconstructed

In [ ]:
print("\n" + "="*60)
print("VISUAL COMPARISONS")
print("="*60)

# MiniBatch DL
print("\nMiniBatch Dictionary Learning:")
fig_mb = plot_reconstruction_comparison(
    X_test_subset, results['minibatch']['reconstructed'],
    method_name="MiniBatch DL"
)
plt.show()

# K-SVD DL (if available)
if results['ksvd'] is not None:
    print("\nK-SVD Dictionary Learning:")
    fig_ksvd = plot_reconstruction_comparison(
        X_test_subset, results['ksvd']['reconstructed'],
        method_name="K-SVD DL"
    )
    plt.show()

# Classical Features
print("\nClassical Features (Pseudo-Inverse):")
fig_classical = plot_reconstruction_comparison(
    X_test_subset, results['classical']['reconstructed'],
    method_name="Classical Features"
)
plt.show()

## 4. Metrics Comparison

In [ ]:
print("\n" + "="*60)
print("METRICS COMPARISON")
print("="*60)

fig_metrics = plot_reconstruction_metrics_comparison(results)
plt.show()

# Print detailed metrics
print("\nDetailed Metrics:")
print("-"*60)
for method_name, result in results.items():
    if result is not None:
        m = result['metrics']
        print(f"\n{method_name.replace('_', ' ').title()}:")
        print(f"  RMSE: {m['rmse']:.6f}")
        print(f"  MAE:  {m['mae']:.6f}")
        print(f"  R²:   {m['r2']:.4f}  (1.0 = perfect)")
        print(f"  SNR:  {m['snr']:.2f} dB")
        print(f"  PRD:  {m['prd']:.2f}%")
        print(f"  Compression: {result['compression_ratio']:.1f}:1")

## 5. Sparse code visualization

In [ ]:
print("\n" + "="*60)
print("SPARSE CODE PATTERNS")
print("="*60)

# MiniBatch sparse codes
fig_sparse_mb = plot_sparse_code_heatmap(
    results['minibatch']['sparse_codes'],
    title="MiniBatch DL — Sparse Codes (k=5)"
)
plt.show()

if results['ksvd'] is not None:
    fig_sparse_ksvd = plot_sparse_code_heatmap(
        results['ksvd']['sparse_codes'],
        title="K-SVD DL — Sparse Codes (k=5)"
    )
    plt.show()

## 6. Dictionary atom visualization

In [ ]:
print("\nVisualizing MiniBatch dictionary atoms...")
atoms_mb = np.load(os.path.join(LOAD_DIR, f'{prefix}_dict_mb_atoms.npy'))
fig_atoms_mb = plot_dictionary_atoms(atoms_mb, n_atoms=16, 
                                      title="MiniBatch Dictionary Atoms (used for reconstruction)")
plt.show()

if results['ksvd'] is not None:
    print("Visualizing K-SVD dictionary atoms...")
    atoms_ksvd = np.load(os.path.join(LOAD_DIR, f'{prefix}_dict_ksvd_atoms.npy'))
    fig_atoms_ksvd = plot_dictionary_atoms(atoms_ksvd, n_atoms=16,
                                            title="K-SVD Dictionary Atoms (used for reconstruction)")
    plt.show()

## 7. Interpretation guide

In [ ]:
print("\n" + "="*60)
print("HOW TO INTERPRET THESE RESULTS")
print("="*60)
print("""
1. R² SCORE:
   - Closer to 1.0 = better reconstruction
   - Dictionary methods should have R² > 0.8
   - Classical features will have lower R² because they lose temporal info

2. SNR (Signal-to-Noise Ratio):
   - Higher dB = better signal quality
   - > 20 dB is good, > 30 dB is excellent

3. VISUAL COMPARISON:
   - Blue (original) and red dashed (reconstructed) should overlap closely
   - Dictionary methods preserve signal shape
   - Classical features may miss high-frequency details

4. SPARSE CODES:
   - Most entries should be white (zero)
   - Only k=5 entries per row should be colored
   - Different windows should use different atom combinations

5. DICTIONARY ATOMS:
   - Each atom is a learned signal pattern
   - They should look like EMG-like waveforms
   - Different atoms capture different aspects of the signal
""")